# WanGP Kaggle Testing Notebook

Run these cells from top to bottom in a Kaggle notebook with GPU enabled. The repo, model checkpoints, Hugging Face cache, Torch cache, pip cache, and temp files are placed under `/kaggle/temp` so `/kaggle/working` is only used for final outputs.

## Cell 1 - Check GPU And Disk

In [ ]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null || df -h

## Cell 2 - Configure Paths

Change `REPO_URL` if you want to test a different fork or branch. Keep the large paths in temp unless you intentionally want Kaggle to persist them.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/SohailAwg/wan2gp_studio.git"
BRANCH = "main"

TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
ROOT = TEMP_ROOT / "Wan2GP-main"
MODEL_DIR = TEMP_ROOT / "wan2gp_models"
CACHE_DIR = TEMP_ROOT / "wan2gp_cache"
OUTPUT_DIR = Path("/kaggle/working/wan2gp_outputs")

# Set False after the first successful install in the same Kaggle session.
INSTALL_DEPS = True

# Safe first-run settings for Kaggle. If Sage/Flash attention is installed later, you can try changing ATTENTION.
ATTENTION = "sdpa"
PROFILE = "4"
PORT = 7860

for path in [TEMP_ROOT, MODEL_DIR, CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo:", ROOT)
print("Models/checkpoints:", MODEL_DIR)
print("Caches:", CACHE_DIR)
print("Persisted outputs:", OUTPUT_DIR)

## Cell 3 - Force Large Caches Into Temp

Run this before installing packages or importing WanGP.

In [ ]:
import os

cache_map = {
    "HF_HOME": CACHE_DIR / "huggingface",
    "HUGGINGFACE_HUB_CACHE": CACHE_DIR / "huggingface" / "hub",
    "TRANSFORMERS_CACHE": CACHE_DIR / "huggingface" / "transformers",
    "DIFFUSERS_CACHE": CACHE_DIR / "huggingface" / "diffusers",
    "TORCH_HOME": CACHE_DIR / "torch",
    "XDG_CACHE_HOME": CACHE_DIR / "xdg",
    "PIP_CACHE_DIR": CACHE_DIR / "pip",
    "GRADIO_TEMP_DIR": TEMP_ROOT / "gradio_tmp",
    "MPLBACKEND": "Agg",
    "TMPDIR": TEMP_ROOT / "tmp",
    "TEMP": TEMP_ROOT / "tmp",
    "TMP": TEMP_ROOT / "tmp",
}

for key, value in cache_map.items():
    if isinstance(value, Path):
        value.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(value)

# Avoid writing tokenizers parallelism warnings into long logs.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

for key in cache_map:
    print(f"{key}={os.environ[key]}")

## Cell 4 - Clone Or Update WanGP In Temp

In [ ]:
import os
import subprocess
import shutil

if (ROOT / ".git").exists():
    os.chdir(ROOT)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if ROOT.exists():
        print(f"Removing incomplete non-git folder: {ROOT}")
        shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)

print("Current repo:", ROOT)
!git -C "$ROOT" rev-parse --short HEAD

## Cell 5 - Link Checkpoints To Temp And Outputs To Working

WanGP normally uses `ckpts` for downloaded model files. This cell makes that folder point at `/kaggle/temp/wan2gp_models`.

In [ ]:
import os
import shutil
from pathlib import Path

def replace_with_symlink(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        if link_path.resolve() == target_path.resolve():
            return
        link_path.unlink()
    elif link_path.exists():
        backup = link_path.with_name(link_path.name + "_backup_in_temp")
        if backup.exists():
            shutil.rmtree(backup) if backup.is_dir() else backup.unlink()
        shutil.move(str(link_path), str(backup))
        print(f"Moved existing {link_path} to {backup}")
    os.symlink(target_path, link_path, target_is_directory=True)

replace_with_symlink(ROOT / "ckpts", MODEL_DIR)
replace_with_symlink(ROOT / "outputs", OUTPUT_DIR)

os.chdir(ROOT)
print("Working directory:", Path.cwd())
print("ckpts ->", (ROOT / "ckpts").resolve())
print("outputs ->", (ROOT / "outputs").resolve())

# Kaggle is headless. This bundled optional mask tool forces TkAgg unless patched.
matanyone_file = ROOT / "preprocessing" / "matanyone" / "tools" / "interact_tools.py"
if matanyone_file.exists():
    text = matanyone_file.read_text(encoding="utf-8")
    patched = text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
    if patched != text:
        matanyone_file.write_text(patched, encoding="utf-8")
        print("Patched MatAnyone matplotlib backend for Kaggle headless mode.")

## Cell 6 - Install Dependencies

This can take a while. It uses `--no-cache-dir` and the temp pip cache so package downloads do not consume `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys

os.chdir(ROOT)

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "--no-cache-dir"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-r", "requirements.txt"], check=True)
else:
    print("Skipping dependency install because INSTALL_DEPS=False")

## Cell 7 - Optional Hugging Face Login

If you need gated models, add a Kaggle secret named `HF_TOKEN` first. Public models can download without this.

In [ ]:
import os

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for name in ["HF_TOKEN", "HUGGINGFACE_TOKEN"]:
            try:
                token = secrets.get_secret(name)
                if token:
                    os.environ["HF_TOKEN"] = token
                    break
            except Exception:
                pass
    except Exception as exc:
        print("Kaggle secrets are not available in this environment:", exc)

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded and Hugging Face login completed.")
else:
    print("No HF_TOKEN found. Public models may still work; gated models will fail until you add the secret.")

## Cell 8 - Launch WanGP UI

This starts WanGP in the background and prints the Gradio public URL when it appears. First model use may download many GB into `/kaggle/temp/wan2gp_models`.

In [ ]:
import os
import re
import subprocess
import sys
import threading
import time

os.chdir(ROOT)

try:
    if WAN2GP_PROC.poll() is None:
        WAN2GP_PROC.terminate()
        time.sleep(5)
except NameError:
    pass

cmd = [
    sys.executable,
    "wgp.py",
    "--share",
    "--listen",
    "--server-port", str(PORT),
    "--attention", ATTENTION,
    "--profile", PROFILE,
    "--perc-reserved-mem-max", "0.25",
]

print("Launching:", " ".join(cmd))
WAN2GP_LOG_LINES = []
WAN2GP_PROC = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

def pump_logs():
    for line in WAN2GP_PROC.stdout:
        WAN2GP_LOG_LINES.append(line.rstrip())
        print(line, end="")

threading.Thread(target=pump_logs, daemon=True).start()

public_url = None
for _ in range(240):
    text = "\n".join(WAN2GP_LOG_LINES[-80:])
    match = re.search(r"https://[a-zA-Z0-9.-]+\.gradio\.live", text)
    if match:
        public_url = match.group(0)
        break
    if WAN2GP_PROC.poll() is not None:
        break
    time.sleep(1)

if public_url:
    print("\nWanGP public URL:", public_url)
else:
    print("\nNo public URL found yet. Check the log above; the process return code is", WAN2GP_PROC.poll())

## Cell 9 - Check Space While Testing

In [ ]:
!du -sh "$ROOT" "$MODEL_DIR" "$CACHE_DIR" "$OUTPUT_DIR" 2>/dev/null || true
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null || df -h

## Cell 10 - List Persisted Outputs

In [ ]:
from pathlib import Path

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path, f"{path.stat().st_size / (1024 ** 2):.1f} MB")

## Cell 11 - Stop WanGP

In [ ]:
try:
    if WAN2GP_PROC.poll() is None:
        WAN2GP_PROC.terminate()
        print("WanGP stopped.")
    else:
        print("WanGP was already stopped with code", WAN2GP_PROC.poll())
except NameError:
    print("WAN2GP_PROC is not defined. Nothing to stop.")